# 03 - Duplicate Pattern Analysis

## Objective

Menganalisis pola duplicate setelah Data Quality Assessment tanpa melakukan fuzzy matching atau mengubah raw dataset.

## Research Questions

1. Apakah exact duplicate sama dengan seluruh record yang memiliki `customer_id` berulang?
2. Field apa yang menghasilkan collision deterministic setelah normalisasi analisis?
3. Berapa banyak candidate pair yang dihasilkan setiap blocking key?
4. Field mana yang layak dipakai pada eksperimen deterministic matching berikutnya?

## Hypothesis

- Exact duplicate hanya menjelaskan sebagian pola duplicate.
- `customer_id` yang berulang dapat memiliki perubahan pada satu atau beberapa field.
- Blocking dengan field high-cardinality akan menghasilkan candidate pair yang jauh lebih kecil daripada perbandingan N x N.

## Scope and privacy

- Raw CSV hanya dibaca.
- Normalisasi dibuat pada salinan analisis dan tidak disimpan kembali ke raw dataset.
- Output hanya berupa hitungan dan ringkasan; nilai customer, email, telepon, dan alamat tidak ditampilkan.

In [1]:
from pathlib import Path
import pandas as pd

DATA_CANDIDATES = [
    Path.cwd() / 'data' / 'raw' / 'crm_50000_customers_dirty_v3.csv',
    Path.cwd().parent / 'data' / 'raw' / 'crm_50000_customers_dirty_v3.csv',
]
DATA_PATH = next((path for path in DATA_CANDIDATES if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError('CSV dataset tidak ditemukan.')

df = pd.read_csv(DATA_PATH)
print(f'File: {DATA_PATH}')
print(f'Shape: {df.shape}')

File: c:\Users\User\Documents\Maganghub 2026\Bulan 1\Tes duplikasi\data\raw\crm_50000_customers_dirty_v3.csv
Shape: (50000, 14)


## Experiment 1 - Exact duplicate versus repeated identifier

Exact duplicate dihitung pada seluruh kolom. Repeated identifier dihitung berdasarkan frekuensi `customer_id`; tidak ada baris yang dihapus.

In [2]:
exact_duplicate_mask = df.duplicated(keep=False)
exact_duplicate_rows = int(df.duplicated().sum())
exact_duplicate_groups = int(df.loc[exact_duplicate_mask].groupby(list(df.columns), dropna=False).ngroups)

customer_group_sizes = df.groupby('customer_id', dropna=False).size()
repeated_customer_groups = customer_group_sizes[customer_group_sizes > 1]
repeated_customer_rows_after_first = int(df['customer_id'].duplicated().sum())

pattern_summary = pd.DataFrame({
    'metric': [
        'exact_duplicate_rows_after_first',
        'exact_duplicate_groups',
        'repeated_customer_id_groups',
        'repeated_customer_id_rows_after_first',
        'maximum_rows_per_customer_id',
    ],
    'value': [
        exact_duplicate_rows,
        exact_duplicate_groups,
        len(repeated_customer_groups),
        repeated_customer_rows_after_first,
        int(customer_group_sizes.max()),
    ],
})
pattern_summary

,metric,value
0,exact_duplicate_rows_after_first,1021
1,exact_duplicate_groups,992
2,repeated_customer_id_groups,1734
3,repeated_customer_id_rows_after_first,1800
4,maximum_rows_per_customer_id,4


## Experiment 2 - Field collision after analysis-only normalization

Normalisasi di bawah hanya dipakai untuk menguji collision deterministic. Ini bukan cleaning permanen.

- Email: trim dan casefold.
- Telepon: ambil digit saja.
- Nama: gabungkan nama depan dan belakang, lowercase, hapus karakter non-alphanumeric.
- DOB: parse ke tanggal ISO.

In [3]:
analysis_df = df.copy()
analysis_df['email_key'] = analysis_df['email'].astype('string').str.strip().str.casefold()
analysis_df['email_key'] = analysis_df['email_key'].replace('', pd.NA)
analysis_df['phone_key'] = analysis_df['phone_number'].astype('string').str.replace(r'\D', '', regex=True)
analysis_df['phone_key'] = analysis_df['phone_key'].replace('', pd.NA)
analysis_df['name_key'] = (
    analysis_df['first_name'].astype('string').fillna('') + ' ' +
    analysis_df['last_name'].astype('string').fillna('')
).str.casefold().str.replace(r'[^a-z0-9]', '', regex=True)
analysis_df['name_key'] = analysis_df['name_key'].replace('', pd.NA)
analysis_df['dob_key'] = pd.to_datetime(analysis_df['dob'], errors='coerce').dt.strftime('%Y-%m-%d')

blocking_keys = ['email_key', 'phone_key', 'name_key', 'dob_key']
collision_rows = []
for key in blocking_keys:
    counts = analysis_df[key].dropna().value_counts()
    repeated = counts[counts > 1]
    collision_rows.append({
        'blocking_key': key,
        'non_null_rows': int(analysis_df[key].notna().sum()),
        'unique_keys': int(analysis_df[key].nunique(dropna=True)),
        'repeated_key_groups': int(len(repeated)),
        'rows_in_repeated_groups': int(repeated.sum()),
        'candidate_pairs': int((repeated * (repeated - 1) // 2).sum()),
        'largest_group': int(repeated.max()) if len(repeated) else 0,
    })

collision_summary = pd.DataFrame(collision_rows).sort_values('candidate_pairs', ascending=False)
collision_summary

,blocking_key,non_null_rows,unique_keys,repeated_key_groups,rows_in_repeated_groups,candidate_pairs,largest_group
3,dob_key,50000,17733,13843,46110,66492,12
2,name_key,50000,40254,6098,15844,18673,22
1,phone_key,50000,46777,2161,5384,5509,9
0,email_key,48960,46363,2102,4699,3422,10


## Experiment 3 - Variation inside repeated customer_id groups

Untuk setiap `customer_id` yang berulang, dihitung berapa banyak nilai berbeda pada field identitas. Hanya ukuran agregat yang ditampilkan.

In [4]:
identity_columns = [
    column for column in [
        'first_name', 'last_name', 'email', 'phone_number',
        'dob', 'address', 'city', 'state', 'country', 'device_id(s)', 'source'
    ]
    if column in df.columns
]

repeated_df = df[df['customer_id'].isin(repeated_customer_groups.index)]
variation_counts = repeated_df.groupby('customer_id')[identity_columns].nunique(dropna=True)
variation_flags = variation_counts.gt(1)

variation_summary = pd.DataFrame({
    'field': identity_columns,
    'repeated_id_groups_with_variation': [int(variation_flags[column].sum()) for column in identity_columns],
    'repeated_id_groups_without_variation': [int((~variation_flags[column]).sum()) for column in identity_columns],
})
variation_summary['total_repeated_id_groups'] = len(variation_counts)
variation_summary['variation_percentage'] = (
    variation_summary['repeated_id_groups_with_variation']
    .div(variation_summary['total_repeated_id_groups'])
    .mul(100)
)
variation_summary.sort_values('variation_percentage', ascending=False).round(2)

,field,repeated_id_groups_with_variation,repeated_id_groups_without_variation,total_repeated_id_groups,variation_percentage
0,first_name,757,977,1734,43.66
1,last_name,757,977,1734,43.66
2,email,0,1734,1734,0.00
3,phone_number,0,1734,1734,0.00
4,dob,0,1734,1734,0.00
5,address,0,1734,1734,0.00
6,city,0,1734,1734,0.00
7,state,0,1734,1734,0.00
8,country,0,1734,1734,0.00
9,device_id(s),0,1734,1734,0.00


## Experiment 4 - Candidate pair size without N x N comparison

Candidate pair dihitung dengan rumus `n * (n - 1) / 2` di dalam setiap repeated blocking group. Tidak ada cross join seluruh dataset.

Kombinasi `name_key + dob_key` juga dihitung karena nama saja dan DOB saja dapat terlalu umum untuk dijadikan kunci tunggal.

In [5]:
analysis_df['name_dob_key'] = analysis_df[['name_key', 'dob_key']].astype('string').fillna('<MISSING>').agg('|'.join, axis=1)
candidate_key_definitions = {
    'email_key': ['email_key'],
    'phone_key': ['phone_key'],
    'name_dob_key': ['name_dob_key'],
    'email_phone_key': ['email_key', 'phone_key'],
}

candidate_rows = []
for label, columns in candidate_key_definitions.items():
    keys = analysis_df[columns].dropna().astype('string').agg('|'.join, axis=1)
    counts = keys.value_counts()
    repeated = counts[counts > 1]
    candidate_rows.append({
        'candidate_rule': label,
        'repeated_blocks': int(len(repeated)),
        'candidate_pairs': int((repeated * (repeated - 1) // 2).sum()),
        'largest_block': int(repeated.max()) if len(repeated) else 0,
    })

candidate_pair_summary = pd.DataFrame(candidate_rows).sort_values('candidate_pairs')
candidate_pair_summary

,candidate_rule,repeated_blocks,candidate_pairs,largest_block
2,name_dob_key,1388,1482,3
3,email_phone_key,1695,1826,4
0,email_key,2102,3422,10
1,phone_key,2161,5509,9


# Result, Analysis, and Decision

Gunakan output aktual sel-sel di atas untuk membaca hasil. Notebook tidak menganggap semua collision sebagai duplicate entity.

## Decision rule

- Exact duplicate dapat diperlakukan sebagai pola yang paling kuat untuk diaudit, tetapi tidak otomatis menjadi keputusan master record.
- Repeated `customer_id` harus dianalisis berdasarkan perubahan field, bukan langsung dihapus atau digabung.
- Blocking key dipilih berdasarkan candidate pair yang manageable dan risiko collision yang dapat dijelaskan.
- Fuzzy matching belum dijalankan karena deterministic pattern dan standardization belum dievaluasi secara terpisah.
- Tidak ada precision, recall, atau metode terbaik yang dapat disimpulkan tanpa ground truth.

## Next Experiment

Jika output menunjukkan format field perlu diseragamkan, tahap berikutnya adalah `04_standardization.ipynb`.
Standardization harus menyimpan kolom turunan dan mempertahankan semua kolom raw. Deterministic matching baru dilakukan setelah aturan standardisasi dan pola candidate pair disepakati.

## Limitations

- Tidak tersedia ground truth pada tahap ini.
- Collision pada satu field tidak membuktikan dua baris adalah customer yang sama.
- Pemeriksaan ini belum menguji typo atau kemiripan fuzzy.
- Hasil hanya berlaku untuk snapshot CSV yang dibaca saat notebook dijalankan.